In [1]:
!pip install pyspark
!sudo apt-get update && sudo apt-get install -y openjdk-17-jdk
import sys
sys.path.insert(0, '/home/jovyan/work/Data-Lakehouse')
from ingesta import get_spark
spark = get_spark()
print(spark.version)

Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:2 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:4 http://archive.ubuntu.com/ubuntu jammy-backports InRelease   
Get:5 http://archive.ubuntu.com/ubuntu jammy-updates/restricted amd64 Packages [7,257 kB]
Get:6 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,940 kB]
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 Packages [4,273 kB]
Get:8 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [7,041 kB]
Fetched 22.8 MB in 2s (11.0 MB/s)                          
Reading package lists... Done
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
openjdk-17-jdk is already the newest version (17.0.18+8-1~22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 149 not upgraded.


:: loading settings :: url = jar:file:/opt/conda/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/jovyan/.ivy2.5.2/cache
The jars for the packages stored in: /home/jovyan/.ivy2.5.2/jars
org.apache.iceberg#iceberg-spark-runtime-4.0_2.13 added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-5273075e-bc4e-4d4d-bc6e-f236568185a6;1.0
	confs: [default]
	found org.apache.iceberg#iceberg-spark-runtime-4.0_2.13;1.10.1 in central
	found org.apache.hadoop#hadoop-aws;3.4.2 in central
	found software.amazon.awssdk#bundle;2.29.52 in central
	found software.amazon.s3.analyticsaccelerator#analyticsaccelerator-s3;1.2.1 in central
	found org.wildfly.openssl#wildfly-openssl;2.1.4.Final in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.367 in central
:: resolution report :: re

Extensions: org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions
Catalog players: org.apache.iceberg.spark.SparkCatalog
4.1.1


In [ ]:
spark.sql("DROP TABLE IF EXISTS players.thesportsdb.bench_nb_cow PURGE")
spark.sql("DROP TABLE IF EXISTS players.thesportsdb.bench_nb_mor PURGE")

In [2]:
import importlib
import benchmark_cow_vs_mor
importlib.reload(benchmark_cow_vs_mor)

<module 'benchmark_cow_vs_mor' from '/home/jovyan/work/Data-Lakehouse/benchmark_cow_vs_mor.py'>

In [5]:
players = benchmark_cow_vs_mor.run([
"--dataset", "thesportsdb",
"--entity", "players",
"--namespace", "thesportsdb",
"--base-table", "bench_nb_players",
"--key-col", "idPlayer",
"--sample-rows", "200000",
"--update-ratio", "0.5",
"--keep-tables"
])

import pandas as pd

rows = [
    ("INSERT", players["by_operation"]["initial_load"]["cow_seconds"],  players["by_operation"]["initial_load"]["mor_seconds"],  players["rows"]),
    ("UPDATE", players["by_operation"]["merge_update"]["cow_seconds"],  players["by_operation"]["merge_update"]["mor_seconds"],  players["updates_rows"]),
    ("DELETE", players["by_operation"]["delete_10pct"]["cow_seconds"],  players["by_operation"]["delete_10pct"]["mor_seconds"],  players["deletes_rows"]),
    ("MERGE",  players["by_operation"]["merge_op"]["cow_seconds"],      players["by_operation"]["merge_op"]["mor_seconds"],      players["merge_rows"]),
    ("SELECT", players["by_operation"]["read_count"]["cow_seconds"],    players["by_operation"]["read_count"]["mor_seconds"],    players["rows"]),
]
df = pd.DataFrame(rows, columns=["Query", "Copy-on-Write", "Merge-on-Read", "Rows"])
df["Copy-on-Write"] = df["Copy-on-Write"].round(3)
df["Merge-on-Read"] = df["Merge-on-Read"].round(3)
df

Extensions: org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions
Catalog players: org.apache.iceberg.spark.SparkCatalog
=== Config ===
source_table: players.thesportsdb.players_bronce
key_col: idPlayer
update_col: dateBorn
update_type: string
sample_rows: 500
updates_rows: 253
deletes_rows: 64
merge_rows:   253

=== Benchmark Results (seconds) ===
players.thesportsdb.bench_nb_players_cow initial_load           0.476
players.thesportsdb.bench_nb_players_cow merge_update           0.941
players.thesportsdb.bench_nb_players_cow delete_10pct           0.823
players.thesportsdb.bench_nb_players_cow merge_op               1.417
players.thesportsdb.bench_nb_players_cow read_count             0.061
players.thesportsdb.bench_nb_players_mor initial_load           0.372
players.thesportsdb.bench_nb_players_mor merge_update           0.796
players.thesportsdb.bench_nb_players_mor delete_10pct           0.744
players.thesportsdb.bench_nb_players_mor merge_op               1.106
players

,Query,Copy-on-Write,Merge-on-Read,Rows
0,INSERT,0.476,0.372,500
1,UPDATE,0.941,0.796,253
2,DELETE,0.823,0.744,64
3,MERGE,1.417,1.106,253
4,SELECT,0.061,0.373,500


In [12]:
teams = benchmark_cow_vs_mor.run([
"--dataset", "thesportsdb",
"--entity", "teams",
"--namespace", "thesportsdb",
"--base-table", "bench_nb_teams",
"--key-col", "idTeam",
"--sample-rows", "200000",
"--update-ratio", "1",
"--keep-tables"
])
rows = [
    ("INSERT", teams["by_operation"]["initial_load"]["cow_seconds"],  teams["by_operation"]["initial_load"]["mor_seconds"],  teams["rows"]),
    ("UPDATE", teams["by_operation"]["merge_update"]["cow_seconds"],  teams["by_operation"]["merge_update"]["mor_seconds"],  teams["updates_rows"]),
    ("DELETE", teams["by_operation"]["delete_10pct"]["cow_seconds"],  teams["by_operation"]["delete_10pct"]["mor_seconds"],  teams["deletes_rows"]),
    ("MERGE",  teams["by_operation"]["merge_op"]["cow_seconds"],      teams["by_operation"]["merge_op"]["mor_seconds"],      teams["merge_rows"]),
    ("SELECT", teams["by_operation"]["read_count"]["cow_seconds"],    teams["by_operation"]["read_count"]["mor_seconds"],    teams["rows"]),
]

df = pd.DataFrame(rows, columns=["Query", "Copy-on-Write", "Merge-on-Read", "Rows"])
df["Copy-on-Write"] = df["Copy-on-Write"].round(3)
df["Merge-on-Read"] = df["Merge-on-Read"].round(3)
df

Extensions: org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions
Catalog players: org.apache.iceberg.spark.SparkCatalog
=== Config ===
source_table: players.thesportsdb.teams_bronce
key_col: idTeam
update_col: idAPIfootball
update_type: string
sample_rows: 1
updates_rows: 1
deletes_rows: 0
merge_rows:   1



=== Benchmark Results (seconds) ===
players.thesportsdb.bench_nb_teams_cow initial_load             0.261
players.thesportsdb.bench_nb_teams_cow merge_update             1.135
players.thesportsdb.bench_nb_teams_cow delete_10pct             0.404
players.thesportsdb.bench_nb_teams_cow merge_op                 1.558
players.thesportsdb.bench_nb_teams_cow read_count               0.349
players.thesportsdb.bench_nb_teams_mor initial_load             0.307
players.thesportsdb.bench_nb_teams_mor merge_update             0.880
players.thesportsdb.bench_nb_teams_mor delete_10pct             0.507
players.thesportsdb.bench_nb_teams_mor merge_op                 1.618
players.thesportsdb.bench_nb_teams_mor read_count               0.405

=== Snapshot Count ===
players.thesportsdb.bench_nb_teams_cow: 5
players.thesportsdb.bench_nb_teams_mor: 5

=== Summary by operation ===
initial_load         COW=   0.261s  MOR=   0.307s  faster=COW (1.18x)
merge_update         COW=   1.135s  MOR=   0.880s  fast

,Query,Copy-on-Write,Merge-on-Read,Rows
0,INSERT,0.261,0.307,1
1,UPDATE,1.135,0.880,1
2,DELETE,0.404,0.507,0
3,MERGE,1.558,1.618,1
4,SELECT,0.349,0.405,1


In [6]:
leagues = benchmark_cow_vs_mor.run([
"--dataset", "thesportsdb",
"--entity", "leagues",
"--namespace", "thesportsdb",
"--base-table", "bench_nb_leagues",
"--key-col", "idLeague",
"--sample-rows", "200000",
"--update-ratio", "0.5",
"--keep-tables"
])
rows = [
    ("INSERT", leagues["by_operation"]["initial_load"]["cow_seconds"],  leagues["by_operation"]["initial_load"]["mor_seconds"],  leagues["rows"]),
    ("UPDATE", leagues["by_operation"]["merge_update"]["cow_seconds"],  leagues["by_operation"]["merge_update"]["mor_seconds"],  leagues["updates_rows"]),
    ("DELETE", leagues["by_operation"]["delete_10pct"]["cow_seconds"],  leagues["by_operation"]["delete_10pct"]["mor_seconds"],  leagues["deletes_rows"]),
    ("MERGE",  leagues["by_operation"]["merge_op"]["cow_seconds"],      leagues["by_operation"]["merge_op"]["mor_seconds"],      leagues["merge_rows"]),
    ("SELECT", leagues["by_operation"]["read_count"]["cow_seconds"],    leagues["by_operation"]["read_count"]["mor_seconds"],    leagues["rows"]),
]
df = pd.DataFrame(rows, columns=["Query", "Copy-on-Write", "Merge-on-Read", "Rows"])
df["Copy-on-Write"] = df["Copy-on-Write"].round(3)
df["Merge-on-Read"] = df["Merge-on-Read"].round(3)
df

Extensions: org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions
Catalog players: org.apache.iceberg.spark.SparkCatalog
=== Config ===
source_table: players.thesportsdb.leagues_bronce
key_col: idLeague
update_col: dateFirstEvent
update_type: string
sample_rows: 5
updates_rows: 1
deletes_rows: 1
merge_rows:   1



=== Benchmark Results (seconds) ===
players.thesportsdb.bench_nb_leagues_cow initial_load           0.496
players.thesportsdb.bench_nb_leagues_cow merge_update           0.609
players.thesportsdb.bench_nb_leagues_cow delete_10pct           0.799
players.thesportsdb.bench_nb_leagues_cow merge_op               0.878
players.thesportsdb.bench_nb_leagues_cow read_count             0.104
players.thesportsdb.bench_nb_leagues_mor initial_load           0.552
players.thesportsdb.bench_nb_leagues_mor merge_update           0.577
players.thesportsdb.bench_nb_leagues_mor delete_10pct           0.461
players.thesportsdb.bench_nb_leagues_mor merge_op               0.870
players.thesportsdb.bench_nb_leagues_mor read_count             0.492

=== Snapshot Count ===
players.thesportsdb.bench_nb_leagues_cow: 5
players.thesportsdb.bench_nb_leagues_mor: 5

=== Summary by operation ===
initial_load         COW=   0.496s  MOR=   0.552s  faster=COW (1.11x)
merge_update         COW=   0.609s  MOR=   0.577s  

,Query,Copy-on-Write,Merge-on-Read,Rows
0,INSERT,0.496,0.552,5
1,UPDATE,0.609,0.577,1
2,DELETE,0.799,0.461,1
3,MERGE,0.878,0.870,1
4,SELECT,0.104,0.492,5


In [ ]:
spark.read.table("players.thesportsdb.bench_nb_leagues_cow").count()

In [ ]:
spark.read.table("players.thesportsdb.bench_nb_leagues_mor").count()

In [ ]:
spark.read.table("players.thesportsdb.bench_nb_teams_cow").count()

In [ ]:
spark.read.table("players.thesportsdb.bench_nb_teams_mor").count()

In [ ]:
spark.read.table("players.thesportsdb.bench_nb_players_cow").count()

In [2]:
import importlib
import benchmark_cow_vs_mor
importlib.reload(benchmark_cow_vs_mor)

<module 'benchmark_cow_vs_mor' from '/home/jovyan/work/Data-Lakehouse/benchmark_cow_vs_mor.py'>

In [5]:
players = benchmark_cow_vs_mor.run([
"--dataset", "transfermarkt",
"--entity", "players",
"--namespace", "transfermarkt",
"--base-table", "bench_nb_players",
"--key-col", "id",
"--sample-rows", "200000",
"--update-ratio", "0.5",
"--keep-tables"
])

import pandas as pd

rows = [
    ("INSERT", players["by_operation"]["initial_load"]["cow_seconds"],  players["by_operation"]["initial_load"]["mor_seconds"],  players["rows"]),
    ("UPDATE", players["by_operation"]["merge_update"]["cow_seconds"],  players["by_operation"]["merge_update"]["mor_seconds"],  players["updates_rows"]),
    ("DELETE", players["by_operation"]["delete_10pct"]["cow_seconds"],  players["by_operation"]["delete_10pct"]["mor_seconds"],  players["deletes_rows"]),
    ("MERGE",  players["by_operation"]["merge_op"]["cow_seconds"],      players["by_operation"]["merge_op"]["mor_seconds"],      players["merge_rows"]),
    ("SELECT", players["by_operation"]["read_count"]["cow_seconds"],    players["by_operation"]["read_count"]["mor_seconds"],    players["rows"]),
]
df = pd.DataFrame(rows, columns=["Query", "Copy-on-Write", "Merge-on-Read", "Rows"])
df["Copy-on-Write"] = df["Copy-on-Write"].round(3)
df["Merge-on-Read"] = df["Merge-on-Read"].round(3)
df

Extensions: org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions
Catalog players: org.apache.iceberg.spark.SparkCatalog
=== Config ===
source_table: players.transfermarkt.players_bronce
key_col: id
update_col: dateOfBirth
update_type: string
sample_rows: 2343
updates_rows: 1209
deletes_rows: 227
merge_rows:   1209



=== Benchmark Results (seconds) ===
players.transfermarkt.bench_nb_players_cow initial_load         0.454
players.transfermarkt.bench_nb_players_cow merge_update         1.061
players.transfermarkt.bench_nb_players_cow delete_10pct         0.873
players.transfermarkt.bench_nb_players_cow merge_op             1.513
players.transfermarkt.bench_nb_players_cow read_count           0.109
players.transfermarkt.bench_nb_players_mor initial_load         0.423
players.transfermarkt.bench_nb_players_mor merge_update         0.797
players.transfermarkt.bench_nb_players_mor delete_10pct         0.748
players.transfermarkt.bench_nb_players_mor merge_op             1.411
players.transfermarkt.bench_nb_players_mor read_count           0.548

=== Snapshot Count ===
players.transfermarkt.bench_nb_players_cow: 5
players.transfermarkt.bench_nb_players_mor: 5

=== Summary by operation ===
initial_load         COW=   0.454s  MOR=   0.423s  faster=MOR (1.07x)
merge_update         COW=   1.061s  MOR=   0.79

,Query,Copy-on-Write,Merge-on-Read,Rows
0,INSERT,0.454,0.423,2343
1,UPDATE,1.061,0.797,1209
2,DELETE,0.873,0.748,227
3,MERGE,1.513,1.411,1209
4,SELECT,0.109,0.548,2343


In [6]:
teams = benchmark_cow_vs_mor.run([
"--dataset", "transfermarkt",
"--entity", "teams",
"--namespace", "transfermarkt",
"--base-table", "bench_nb_teams",
"--key-col", "id",
"--sample-rows", "200000",
"--update-ratio", "0.5",
"--keep-tables"
])
rows = [
    ("INSERT", teams["by_operation"]["initial_load"]["cow_seconds"],  teams["by_operation"]["initial_load"]["mor_seconds"],  teams["rows"]),
    ("UPDATE", teams["by_operation"]["merge_update"]["cow_seconds"],  teams["by_operation"]["merge_update"]["mor_seconds"],  teams["updates_rows"]),
    ("DELETE", teams["by_operation"]["delete_10pct"]["cow_seconds"],  teams["by_operation"]["delete_10pct"]["mor_seconds"],  teams["deletes_rows"]),
    ("MERGE",  teams["by_operation"]["merge_op"]["cow_seconds"],      teams["by_operation"]["merge_op"]["mor_seconds"],      teams["merge_rows"]),
    ("SELECT", teams["by_operation"]["read_count"]["cow_seconds"],    teams["by_operation"]["read_count"]["mor_seconds"],    teams["rows"]),
]

df = pd.DataFrame(rows, columns=["Query", "Copy-on-Write", "Merge-on-Read", "Rows"])
df["Copy-on-Write"] = df["Copy-on-Write"].round(3)
df["Merge-on-Read"] = df["Merge-on-Read"].round(3)
df

Extensions: org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions
Catalog players: org.apache.iceberg.spark.SparkCatalog
=== Config ===
source_table: players.transfermarkt.teams_bronce
key_col: id
update_col: addressLine1
update_type: string
sample_rows: 95
updates_rows: 40
deletes_rows: 7
merge_rows:   40

=== Benchmark Results (seconds) ===
players.transfermarkt.bench_nb_teams_cow initial_load           0.430
players.transfermarkt.bench_nb_teams_cow merge_update           0.673
players.transfermarkt.bench_nb_teams_cow delete_10pct           0.762
players.transfermarkt.bench_nb_teams_cow merge_op               0.947
players.transfermarkt.bench_nb_teams_cow read_count             0.151
players.transfermarkt.bench_nb_teams_mor initial_load           0.450
players.transfermarkt.bench_nb_teams_mor merge_update           0.518
players.transfermarkt.bench_nb_teams_mor delete_10pct           0.633
players.transfermarkt.bench_nb_teams_mor merge_op               0.805
players.trans

,Query,Copy-on-Write,Merge-on-Read,Rows
0,INSERT,0.430,0.450,95
1,UPDATE,0.673,0.518,40
2,DELETE,0.762,0.633,7
3,MERGE,0.947,0.805,40
4,SELECT,0.151,0.391,95


In [13]:
leagues = benchmark_cow_vs_mor.run([
"--dataset", "transfermarkt",
"--entity", "leagues",
"--namespace", "transfermarkt",
"--base-table", "bench_nb_leagues",
"--key-col", "id",
"--sample-rows", "200000",
"--update-ratio", "1",
"--keep-tables"
])
rows = [
    ("INSERT", leagues["by_operation"]["initial_load"]["cow_seconds"],  leagues["by_operation"]["initial_load"]["mor_seconds"],  leagues["rows"]),
    ("UPDATE", leagues["by_operation"]["merge_update"]["cow_seconds"],  leagues["by_operation"]["merge_update"]["mor_seconds"],  leagues["updates_rows"]),
    ("DELETE", leagues["by_operation"]["delete_10pct"]["cow_seconds"],  leagues["by_operation"]["delete_10pct"]["mor_seconds"],  leagues["deletes_rows"]),
    ("MERGE",  leagues["by_operation"]["merge_op"]["cow_seconds"],      leagues["by_operation"]["merge_op"]["mor_seconds"],      leagues["merge_rows"]),
    ("SELECT", leagues["by_operation"]["read_count"]["cow_seconds"],    leagues["by_operation"]["read_count"]["mor_seconds"],    leagues["rows"]),
]
df = pd.DataFrame(rows, columns=["Query", "Copy-on-Write", "Merge-on-Read", "Rows"])
df["Copy-on-Write"] = df["Copy-on-Write"].round(3)
df["Merge-on-Read"] = df["Merge-on-Read"].round(3)
df

Extensions: org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions
Catalog players: org.apache.iceberg.spark.SparkCatalog
=== Config ===
source_table: players.transfermarkt.leagues_bronce
key_col: id
update_col: clubs
update_type: numeric
sample_rows: 5
updates_rows: 5
deletes_rows: 0
merge_rows:   5



=== Benchmark Results (seconds) ===
players.transfermarkt.bench_nb_leagues_cow initial_load         0.484
players.transfermarkt.bench_nb_leagues_cow merge_update         1.144
players.transfermarkt.bench_nb_leagues_cow delete_10pct         0.864
players.transfermarkt.bench_nb_leagues_cow merge_op             1.997
players.transfermarkt.bench_nb_leagues_cow read_count           0.502
players.transfermarkt.bench_nb_leagues_mor initial_load         0.467
players.transfermarkt.bench_nb_leagues_mor merge_update         1.109
players.transfermarkt.bench_nb_leagues_mor delete_10pct         0.728
players.transfermarkt.bench_nb_leagues_mor merge_op             1.384
players.transfermarkt.bench_nb_leagues_mor read_count           0.762

=== Snapshot Count ===
players.transfermarkt.bench_nb_leagues_cow: 5
players.transfermarkt.bench_nb_leagues_mor: 5

=== Summary by operation ===
initial_load         COW=   0.484s  MOR=   0.467s  faster=MOR (1.04x)
merge_update         COW=   1.144s  MOR=   1.10

,Query,Copy-on-Write,Merge-on-Read,Rows
0,INSERT,0.484,0.467,5
1,UPDATE,1.144,1.109,5
2,DELETE,0.864,0.728,0
3,MERGE,1.997,1.384,5
4,SELECT,0.502,0.762,5
